### GMM

In [36]:
import pandas as pd
import numpy as np

In [37]:
from sklearn.mixture import GaussianMixture
from scipy.stats import chi2_contingency
from itertools import combinations

In [38]:
# Load data
expression = pd.read_csv("shared_expression_log2.csv", index_col=0)
sample_meta = pd.read_csv("shared_sample_metadata.csv", index_col=0)

In [39]:
print(expression.shape)

(32, 32309)


In [40]:
gene_variance = expression.var(axis=0)
top_5000_genes = gene_variance.nlargest(5000).index
expression_5000 = expression.loc[:, top_5000_genes]

In [41]:
print(expression_5000.shape)

(32, 5000)


In [42]:
# GMM at diff k values
k_values = [2,3,4,5,6]

for k in k_values:
    gmm = GaussianMixture(
        n_components=k,
        covariance_type="diag",
        random_state=42
    )

    sample_meta[f"gmm_k_{k}"] = gmm.fit_predict(expression_5000)

In [43]:
# compare k values
for k1, k2 in zip(k_values[:-1], k_values[1:]):
    print(f"\nk={k1} vs k={k2}")

    print(pd.crosstab(
        sample_meta[f"gmm_k_{k1}"],
        sample_meta[f"gmm_k_{k2}"]
    ))


k=2 vs k=3
gmm_k_3  0   1  2
gmm_k_2          
0        7   0  8
1        0  16  1

k=3 vs k=4
gmm_k_4  0   1  2  3
gmm_k_3             
0        7   0  0  0
1        0  10  0  6
2        0   0  8  1

k=4 vs k=5
gmm_k_5  0   1  2  3  4
gmm_k_4                
0        7   0  0  0  0
1        0  10  0  0  0
2        0   0  8  0  0
3        0   0  0  6  1

k=5 vs k=6
gmm_k_6  0  1  2  3  4  5
gmm_k_5                  
0        7  0  0  0  0  0
1        0  9  0  0  0  1
2        0  0  8  0  0  0
3        0  0  0  4  0  2
4        0  0  0  0  1  0


In [44]:
# Rerun with different # genes
gene_values = [10, 100, 1000, 5000, 10000]

for i in gene_values:
    top_genes = gene_variance.nlargest(i).index
    subset = expression.loc[:, top_genes]

    gmm = GaussianMixture(
        n_components=2,
        covariance_type="diag",
        random_state=42
    )

    sample_meta[f"gmm_{i}_genes"] = gmm.fit_predict(subset)
sample_meta.head()

,condition,age,genotype,gmm_k_2,gmm_k_3,gmm_k_4,gmm_k_5,gmm_k_6,gmm_10_genes,gmm_100_genes,gmm_1000_genes,gmm_5000_genes,gmm_10000_genes
refinebio_title,,,,,,,,,,,,,
"Ground Control, 4 days old, Col-0 Rep1",ground,4d,Col-0,0,2,2,2,2,0,0,0,0,0
"Ground Control, 4 days old, Col-0 Rep2",ground,4d,Col-0,0,2,2,2,2,0,0,0,0,0
"Ground Control, 4 days old, Col-0 Rep3",ground,4d,Col-0,0,2,2,2,2,0,0,0,0,0
"Ground Control, 4 days old, Col-0 Rep4",ground,4d,Col-0,0,2,2,2,2,0,0,0,0,0
"Ground Control, 4 days old, WS Rep1",ground,4d,WS,1,1,1,1,1,1,1,1,1,1


In [45]:
gene_cols = [
    "gmm_10_genes",
    "gmm_100_genes",
    "gmm_1000_genes",
    "gmm_5000_genes",
    "gmm_10000_genes"
]

In [48]:
# chi2
rows = []

for col1, col2 in combinations(gene_cols, 2):
    table = pd.crosstab(
        sample_meta[col1],
        sample_meta[col2]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    rows.append({
        "Clustering_1": col1,
        "Clustering_2": col2,
        "Chi_Square": chi2,
        "DoF": dof,
        "P_Value": p
    })

chi_square_results = pd.DataFrame(rows)

chi_square_results

,Clustering_1,Clustering_2,Chi_Square,DoF,P_Value
0,gmm_10_genes,gmm_100_genes,28.125000,1,1.137273e-07
1,gmm_10_genes,gmm_1000_genes,28.125000,1,1.137273e-07
2,gmm_10_genes,gmm_5000_genes,24.596078,1,7.069546e-07
3,gmm_10_genes,gmm_10000_genes,24.596078,1,7.069546e-07
4,gmm_100_genes,gmm_1000_genes,28.125000,1,1.137273e-07
5,gmm_100_genes,gmm_5000_genes,24.596078,1,7.069546e-07
6,gmm_100_genes,gmm_10000_genes,24.596078,1,7.069546e-07
7,gmm_1000_genes,gmm_5000_genes,24.596078,1,7.069546e-07
8,gmm_1000_genes,gmm_10000_genes,24.596078,1,7.069546e-07
9,gmm_5000_genes,gmm_10000_genes,28.110296,1,1.145946e-07


In [47]:
sample_meta.to_csv("results/gmm_cluster_assignments.csv")
chi_square_results.to_csv("results/gmm_gene_count_chi_square.csv", index=False)